# Stanford RNA 3D Folding Part 2 — Kaggle Submission

Single self-contained notebook: template-based prediction with two-stage matching (k-mer → alignment), Kabsch superposition, and multi-chain assembly. Outputs `submission.csv` to `/kaggle/working/` for submission.

In [ ]:
# --- Imports & config (Kaggle paths, GPU/CPU backend) ---
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Backend: GPU (CuPy) when available (e.g. Kaggle GPU), else CPU (NumPy). Mac uses CPU.
try:
    import cupy as cp
    _ = cp.array([1.0])
    xp = cp
    device = "cuda"
except Exception:
    xp = np
    device = "cpu"
def asnumpy(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)
print("Backend:", device)

# Kaggle: input data and working (output) directory
OUTPUT_DIR = Path("/kaggle/working")
_base = Path("/kaggle/input/stanford-rna-3d-folding-2")
if _base.exists():
    # Data may be in base or in a subfolder (e.g. stanford-rna-3d-folding-2/stanford-rna-3d-folding-2)
    if (_base / "train_sequences.csv").exists():
        INPUT_DIR = _base
    else:
        _found = None
        for _d in _base.iterdir():
            if _d.is_dir() and (_d / "train_sequences.csv").exists():
                _found = _d
                break
        INPUT_DIR = _found if _found is not None else _base
else:
    INPUT_DIR = Path("data")
    OUTPUT_DIR = Path("output")

# Config (always define so prediction cell can use them)
SEED = int(os.environ.get("RNA_FOLDING_SEED", "42"))
np.random.seed(SEED)
KMER_SIZE = 5
PREFILTER_TOP_N = 50
LENGTH_RATIO_MIN = 0.3
MAX_ALIGN_CANDIDATES = 30
LENGTH_RATIO_GLOBAL_THRESHOLD = 0.85
ALIGNMENT_SCORE_WEIGHT = 0.65
COVERAGE_WEIGHT = 0.25
LENGTH_SIMILARITY_WEIGHT = 0.10
EXTRAPOLATION_CAP_RATIO = 1.20
SMOOTHING_WINDOW = 5
MIN_ALIGNMENT_SCORE_FOR_TEMPLATE = 0.05
PERTURBATION_SIGMA = 0.3

In [ ]:
# --- Geometry (uses xp = NumPy or CuPy) ---
RISE_PER_RESIDUE = 2.81
ROTATION_PER_RESIDUE = 32.7 * 3.141592653589793 / 180.0
HELIX_RADIUS = 9.4

def generate_aform_helix(sequence_length, offset=0):
    indices = xp.arange(sequence_length, dtype=xp.float32) + offset
    angles = indices * ROTATION_PER_RESIDUE
    coords = xp.zeros((sequence_length, 3), dtype=xp.float32)
    coords[:, 0] = HELIX_RADIUS * xp.cos(angles)
    coords[:, 1] = HELIX_RADIUS * xp.sin(angles)
    coords[:, 2] = indices * RISE_PER_RESIDUE
    return coords

def _sanitize_coords(coords):
    arr = asnumpy(coords)
    result = arr.copy()
    for col_idx in range(3):
        col = result[:, col_idx]
        bad = ~np.isfinite(col)
        if bad.all():
            result[:, col_idx] = 0.0
            continue
        if bad.any():
            valid_idx = np.where(~bad)[0]
            bad_idx = np.where(bad)[0]
            result[bad_idx, col_idx] = np.interp(bad_idx, valid_idx, col[valid_idx])
    return xp.asarray(result)

def resample_coordinates(coords, target_len):
    arr = asnumpy(coords)
    if len(arr) == target_len:
        return xp.asarray(arr)
    result = np.zeros((target_len, 3), dtype=np.float32)
    for dim in range(3):
        result[:, dim] = np.interp(np.linspace(0, 1, target_len), np.linspace(0, 1, len(arr)), arr[:, dim])
    return xp.asarray(result)

In [ ]:
# --- Alignment (Needleman-Wunsch, Smith-Waterman) ---
MATCH_SCORE = 4
MISMATCH_SCORE = -2
GAP_OPEN = -10
GAP_EXTEND = -1
_PURINES = set("AG")
_PYRIMIDINES = set("CU")

def _substitution_score(a, b):
    if a == b:
        return MATCH_SCORE
    if (a in _PURINES and b in _PURINES) or (a in _PYRIMIDINES and b in _PYRIMIDINES):
        return MISMATCH_SCORE // 2
    return MISMATCH_SCORE

def needleman_wunsch(seq1, seq2):
    n, m = len(seq1), len(seq2)
    NEG_INF = -999999
    M = np.full((n + 1, m + 1), NEG_INF, dtype=np.int32)
    X = np.full((n + 1, m + 1), NEG_INF, dtype=np.int32)
    Y = np.full((n + 1, m + 1), NEG_INF, dtype=np.int32)
    M[0, 0] = 0
    for i in range(1, n + 1):
        X[i, 0] = GAP_OPEN + (i - 1) * GAP_EXTEND
    for j in range(1, m + 1):
        Y[0, j] = GAP_OPEN + (j - 1) * GAP_EXTEND
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = _substitution_score(seq1[i - 1], seq2[j - 1])
            M[i, j] = sub + max(M[i - 1, j - 1], X[i - 1, j - 1], Y[i - 1, j - 1])
            X[i, j] = max(M[i - 1, j] + GAP_OPEN, X[i - 1, j] + GAP_EXTEND)
            Y[i, j] = max(M[i, j - 1] + GAP_OPEN, Y[i, j - 1] + GAP_EXTEND)
    score = max(M[n, m], X[n, m], Y[n, m])
    mapping = _traceback_global(M, X, Y, seq1, seq2)
    return score, mapping

def _traceback_global(M, X, Y, seq1, seq2):
    n, m = len(seq1), len(seq2)
    mapping = []
    i, j = n, m
    scores = [(M[n, m], 0), (X[n, m], 1), (Y[n, m], 2)]
    state = max(scores, key=lambda x: x[0])[1]
    while i > 0 or j > 0:
        if state == 0:
            if i > 0 and j > 0:
                mapping.append((i - 1, j - 1))
                prev_scores = [(M[i - 1, j - 1], 0), (X[i - 1, j - 1], 1), (Y[i - 1, j - 1], 2)]
                state = max(prev_scores, key=lambda x: x[0])[1]
                i -= 1
                j -= 1
            elif i > 0:
                mapping.append((i - 1, -1))
                i -= 1
            else:
                mapping.append((-1, j - 1))
                j -= 1
        elif state == 1:
            mapping.append((i - 1, -1))
            if M[i - 1, j] + GAP_OPEN >= X[i - 1, j] + GAP_EXTEND:
                state = 0
            i -= 1
        else:
            mapping.append((-1, j - 1))
            if M[i, j - 1] + GAP_OPEN >= Y[i, j - 1] + GAP_EXTEND:
                state = 0
            j -= 1
    mapping.reverse()
    return mapping

def smith_waterman(seq1, seq2):
    n, m = len(seq1), len(seq2)
    NEG_INF = -999999
    H = np.zeros((n + 1, m + 1), dtype=np.int32)
    X = np.full((n + 1, m + 1), NEG_INF, dtype=np.int32)
    Y = np.full((n + 1, m + 1), NEG_INF, dtype=np.int32)
    max_score, max_pos = 0, (0, 0)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = _substitution_score(seq1[i - 1], seq2[j - 1])
            H[i, j] = max(0, H[i - 1, j - 1] + sub, X[i - 1, j - 1] + sub, Y[i - 1, j - 1] + sub)
            X[i, j] = max(H[i - 1, j] + GAP_OPEN, X[i - 1, j] + GAP_EXTEND)
            Y[i, j] = max(H[i, j - 1] + GAP_OPEN, Y[i, j - 1] + GAP_EXTEND)
            if H[i, j] > max_score:
                max_score, max_pos = H[i, j], (i, j)
    aligned_pairs = []
    i, j = max_pos
    while i > 0 and j > 0 and H[i, j] > 0:
        if H[i, j] == H[i - 1, j - 1] + _substitution_score(seq1[i - 1], seq2[j - 1]):
            aligned_pairs.append((i - 1, j - 1))
            i -= 1
            j -= 1
        elif X[i, j] >= Y[i, j]:
            i -= 1
        else:
            j -= 1
    aligned_pairs.reverse()
    return max_score, aligned_pairs

def compute_alignment_score(seq1, seq2, mode="auto", length_ratio_global_threshold=0.85):
    len1, len2 = len(seq1), len(seq2)
    if len1 == 0 or len2 == 0:
        return 0.0, []
    length_ratio = min(len1, len2) / max(len1, len2)
    if mode == "auto":
        mode = "global" if length_ratio >= length_ratio_global_threshold else "local"
    if mode == "global":
        score, mapping = needleman_wunsch(seq1, seq2)
        matched = [(i, j) for i, j in mapping if i >= 0 and j >= 0]
    else:
        score, matched = smith_waterman(seq1, seq2)
    max_possible = MATCH_SCORE * min(len1, len2)
    normalized = score / max_possible if max_possible > 0 else 0.0
    return max(0.0, normalized), matched

def get_residue_mapping(aligned_pairs, target_len, template_len):
    mapping = np.full(target_len, -1, dtype=np.int32)
    for t_idx, s_idx in aligned_pairs:
        if 0 <= t_idx < target_len and 0 <= s_idx < template_len:
            mapping[t_idx] = s_idx
    aligned_positions = np.where(mapping >= 0)[0]
    if len(aligned_positions) == 0:
        for i in range(target_len):
            mapping[i] = min(int(i * template_len / target_len), template_len - 1)
    elif len(aligned_positions) < target_len:
        for i in range(target_len):
            if mapping[i] >= 0:
                continue
            left = aligned_positions[aligned_positions < i]
            right = aligned_positions[aligned_positions > i]
            if len(left) > 0 and len(right) > 0:
                li, ri = left[-1], right[0]
                lv, rv = mapping[li], mapping[ri]
                frac = (i - li) / (ri - li)
                mapping[i] = min(int(lv + frac * (rv - lv)), template_len - 1)
            elif len(left) > 0:
                li = left[-1]
                mapping[i] = min(mapping[li] + (i - li), template_len - 1)
            else:
                ri = right[0]
                mapping[i] = max(mapping[ri] - (ri - i), 0)
    return mapping

In [ ]:
# --- Superposition (Kabsch, uses xp) ---
def kabsch_rmsd(P, Q):
    assert P.shape == Q.shape and P.shape[1] == 3
    centroid_P = P.mean(axis=0)
    centroid_Q = Q.mean(axis=0)
    p, q = P - centroid_P, Q - centroid_Q
    H = p.T @ q
    U, S, Vt = xp.linalg.svd(H)
    d = xp.linalg.det(Vt.T @ U.T)
    sign_matrix = xp.diag(xp.array([1.0, 1.0, float(xp.sign(d))], dtype=xp.float32))
    R = Vt.T @ sign_matrix @ U.T
    t = centroid_Q - R @ centroid_P
    p_rotated = p @ R.T
    rmsd = float(xp.sqrt(((p_rotated - q) ** 2).sum() / len(P)))
    return R, t, rmsd

def apply_superposition(coords, R, t):
    return coords @ R.T + t

def superimpose_with_alignment(template_coords, target_coords, template_indices, target_indices):
    P = template_coords[template_indices]
    Q = target_coords[target_indices]
    valid = xp.isfinite(P).all(axis=1) & xp.isfinite(Q).all(axis=1)
    P, Q = P[valid], Q[valid]
    if len(P) < 3:
        return xp.eye(3, dtype=xp.float32), xp.zeros(3, dtype=xp.float32), float('inf')
    return kabsch_rmsd(P, Q)

def compute_tm_score(coords1, coords2):
    L = len(coords1)
    if L == 0:
        return 0.0
    valid = xp.isfinite(coords1).all(axis=1) & xp.isfinite(coords2).all(axis=1)
    if int(valid.sum()) < 3:
        return 0.0
    c1_valid, c2_valid = coords1[valid], coords2[valid]
    R, t, _ = kabsch_rmsd(c1_valid, c2_valid)
    aligned = apply_superposition(c1_valid, R, t)
    d_0 = max(1.24 * max(L - 15, 1) ** (1.0/3.0) - 1.8, 0.5)
    distances = xp.sqrt(((aligned - c2_valid) ** 2).sum(axis=1))
    return float((1.0 / (1.0 + (distances / d_0) ** 2)).sum()) / L

In [ ]:
# --- Multichain ---
def parse_stoichiometry(stoich_str):
    chains = []
    for part in stoich_str.split(";"):
        part = part.strip()
        if ":" in part:
            label, count = part.rsplit(":", 1)
            chains.append((label, int(count)))
        else:
            chains.append((part, 1))
    return chains

def parse_all_sequences_fasta(all_sequences_str):
    if pd.isna(all_sequences_str) or not all_sequences_str.strip():
        return []
    entries = []
    current_header, current_seq_parts = None, []
    for line in all_sequences_str.strip().split("\n"):
        line = line.strip()
        if line.startswith(">"):
            if current_header is not None and current_seq_parts:
                entries.append((current_header, "".join(current_seq_parts)))
            current_header = line[1:]
            current_seq_parts = []
        elif line:
            current_seq_parts.append(line)
    if current_header is not None and current_seq_parts:
        entries.append((current_header, "".join(current_seq_parts)))
    return entries

def get_chain_sequences(full_sequence, stoichiometry, all_sequences_str=None):
    if all_sequences_str and not pd.isna(all_sequences_str):
        fasta_entries = parse_all_sequences_fasta(all_sequences_str)
        if fasta_entries:
            return [(header.split("|")[0] if "|" in header else f"chain_{i}", i, seq)
                    for i, (header, seq) in enumerate(fasta_entries)]
    total_copies = sum(c for _, c in stoichiometry)
    if total_copies <= 0:
        return [("A", 0, full_sequence)]
    unique_chains = set(l for l, _ in stoichiometry)
    if len(unique_chains) == 1:
        label, n_copies = stoichiometry[0]
        chain_len = len(full_sequence) // n_copies
        return [(label, i, full_sequence[i*chain_len:(i+1)*chain_len if i < n_copies-1 else len(full_sequence)])
                for i in range(n_copies)]
    chain_len = len(full_sequence) // total_copies
    result = []
    pos = 0
    for label, count in stoichiometry:
        for i in range(count):
            end = min(pos + chain_len, len(full_sequence))
            result.append((label, i, full_sequence[pos:end]))
            pos = end
    return result

def _rotation_matrix_z(angle):
    c, s = float(xp.cos(angle)), float(xp.sin(angle))
    return xp.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=xp.float32)

def generate_symmetric_copies(single_chain_coords, n_copies, arrangement="circular"):
    L = len(single_chain_coords)
    if n_copies == 1:
        return single_chain_coords.copy()
    centroid = single_chain_coords.mean(axis=0)
    centered = single_chain_coords - centroid
    max_extent = float(xp.max(xp.linalg.norm(centered, axis=1)))
    separation = max_extent * 2.5
    all_coords = xp.zeros((n_copies * L, 3), dtype=xp.float32)
    if arrangement == "circular" and n_copies > 2:
        radius = separation / (2.0 * float(xp.sin(3.141592653589793 / n_copies)))
        for i in range(n_copies):
            angle = 2.0 * 3.141592653589793 * i / n_copies
            R = _rotation_matrix_z(angle)
            offset = xp.array([radius * xp.cos(angle), radius * xp.sin(angle), 0.0], dtype=xp.float32)
            all_coords[i*L:(i+1)*L] = centered @ R.T + centroid + offset
    else:
        z_extent = float(centered[:, 2].max() - centered[:, 2].min())
        stack_distance = max(z_extent * 1.3, separation)
        total_height = (n_copies - 1) * stack_distance
        for i in range(n_copies):
            offset = xp.array([0.0, 0.0, i * stack_distance - total_height / 2], dtype=xp.float32)
            angle = 2.0 * 3.141592653589793 * i / n_copies
            R = _rotation_matrix_z(angle)
            all_coords[i*L:(i+1)*L] = centered @ R.T + centroid + offset
    return all_coords

def assemble_multimer(chain_coords_list, stoichiometry, full_length):
    unique_chains = set(l for l, _ in stoichiometry)
    if len(unique_chains) == 1 and len(chain_coords_list) >= 1:
        _, n_copies = stoichiometry[0]
        single_chain = chain_coords_list[0]
        result = generate_symmetric_copies(single_chain, n_copies) if n_copies > 1 else single_chain
        if len(result) >= full_length:
            return result[:full_length]
        padded = xp.zeros((full_length, 3), dtype=xp.float32)
        padded[:len(result)] = result
        return padded
    concatenated = xp.concatenate(chain_coords_list, axis=0)
    if len(concatenated) >= full_length:
        return concatenated[:full_length]
    padded = xp.zeros((full_length, 3), dtype=xp.float32)
    padded[:len(concatenated)] = concatenated
    return padded

In [ ]:
# --- Data loader ---
def _interpolate_nan(coords):
    for col_idx in range(3):
        col = coords[:, col_idx]
        nan_mask = np.isnan(col)
        if nan_mask.all():
            coords[:, col_idx] = 0.0
            continue
        if nan_mask.any():
            valid_idx = np.where(~nan_mask)[0]
            nan_idx = np.where(nan_mask)[0]
            coords[nan_idx, col_idx] = np.interp(nan_idx, valid_idx, col[valid_idx])

def load_train_sequences(data_dir):
    path = Path(data_dir) / "train_sequences.csv"
    df = pd.read_csv(path, low_memory=False)
    df["seq_len"] = df["sequence"].apply(len)
    return df

def load_test_sequences(data_dir):
    path = Path(data_dir) / "test_sequences.csv"
    df = pd.read_csv(path)
    df["seq_len"] = df["sequence"].apply(len)
    return df

def load_sample_submission(data_dir):
    return pd.read_csv(Path(data_dir) / "sample_submission.csv")

def load_train_labels(data_dir):
    path = Path(data_dir) / "train_labels.csv"
    df = pd.read_csv(path, low_memory=False)
    df["target_id"] = df["ID"].apply(lambda x: x.rsplit("_", 1)[0])
    return df

def build_train_structure_lookup(train_labels):
    lookup = {}
    grouped = train_labels.groupby("target_id")
    for target_id, group in tqdm(grouped, desc="Structures"):
        copies_available = sorted(group["copy"].unique())
        all_copies = {}
        for copy_id in copies_available:
            copy_group = group[group["copy"] == copy_id].sort_values("resid")
            coords = copy_group[["x_1", "y_1", "z_1"]].values.astype(np.float32)
            _interpolate_nan(coords)
            all_copies[copy_id] = xp.asarray(coords)
        primary = group[group["copy"] == copies_available[0]].sort_values("resid")
        lookup[target_id] = {
            "coords": all_copies[copies_available[0]],
            "all_copies": all_copies,
            "resnames": primary["resname"].values.tolist(),
            "resids": primary["resid"].values.tolist(),
            "chains": primary["chain"].values.tolist(),
            "n_copies": len(copies_available),
        }
    return lookup

def parse_submission_targets(sample_sub):
    targets = defaultdict(list)
    for _, row in sample_sub.iterrows():
        target_id = row["ID"].rsplit("_", 1)[0]
        resid = int(row["ID"].rsplit("_", 1)[1])
        targets[target_id].append({"ID": row["ID"], "resname": row["resname"], "resid": resid})
    return dict(targets)

In [ ]:
# --- Template matcher ---
def _parse_cutoff(date_str):
    try:
        from datetime import datetime
        return datetime.strptime(str(date_str).strip()[:10], "%Y-%m-%d")
    except Exception:
        return None

def kmer_prefilter(test_seq, train_sequences, train_structures, allowed_ids=None, k=None, top_n=None, length_ratio_min=None):
    k = k or KMER_SIZE
    top_n = top_n or PREFILTER_TOP_N
    length_ratio_min = length_ratio_min or LENGTH_RATIO_MIN
    test_len = len(test_seq)
    test_kmers = set(test_seq[i:i+k] for i in range(len(test_seq) - k + 1)) if len(test_seq) >= k else set()
    if not test_kmers:
        cands = [x for x in train_structures.keys() if allowed_ids is None or x in allowed_ids]
        return cands[:top_n]
    scores = []
    for train_id, train_seq in train_sequences.items():
        if train_id not in train_structures or (allowed_ids is not None and train_id not in allowed_ids):
            continue
        train_len = len(train_seq)
        if min(test_len, train_len) / max(test_len, train_len) < length_ratio_min:
            continue
        train_kmers = set(train_seq[i:i+k] for i in range(len(train_seq) - k + 1)) if len(train_seq) >= k else set()
        if not train_kmers:
            continue
        inter = len(test_kmers & train_kmers)
        union = len(test_kmers | train_kmers)
        jaccard = inter / union if union > 0 else 0.0
        length_ratio = min(test_len, train_len) / max(test_len, train_len)
        scores.append((train_id, jaccard * 0.6 + length_ratio * 0.4))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [tid for tid, _ in scores[:top_n]]

def rank_templates_by_alignment(test_seq, candidate_ids, train_sequences, max_align=None):
    max_align = max_align or MAX_ALIGN_CANDIDATES
    test_len = len(test_seq)
    results = []
    for train_id in candidate_ids[:max_align]:
        train_seq = train_sequences[train_id]
        train_len = len(train_seq)
        test_sub = test_seq[:2000]
        train_sub = train_seq[:2000]
        score, aligned_pairs = compute_alignment_score(test_sub, train_sub, mode="auto", length_ratio_global_threshold=LENGTH_RATIO_GLOBAL_THRESHOLD)
        coverage = len(set(p[0] for p in aligned_pairs if p[0] >= 0)) / len(test_sub) if aligned_pairs and test_sub else 0.0
        length_sim = min(test_len, train_len) / max(test_len, train_len)
        combined = score * ALIGNMENT_SCORE_WEIGHT + coverage * COVERAGE_WEIGHT + length_sim * LENGTH_SIMILARITY_WEIGHT
        results.append((train_id, combined, aligned_pairs))
    results.sort(key=lambda x: x[1], reverse=True)
    return results

def _fallback_transfer(template_coords, target_len):
    template_len = len(template_coords)
    if template_len == target_len:
        return template_coords.copy()
    if template_len > target_len:
        return template_coords[:target_len].copy()
    arr = asnumpy(template_coords)
    result = np.zeros((target_len, 3), dtype=np.float32)
    for dim in range(3):
        result[:, dim] = np.interp(np.linspace(0, 1, target_len), np.linspace(0, 1, template_len), arr[:, dim])
    return xp.asarray(result)

def transfer_coords_with_alignment(template_coords, aligned_pairs, target_len, template_len, apply_kabsch=True, extrapolation_cap_ratio=None):
    extrapolation_cap_ratio = extrapolation_cap_ratio or EXTRAPOLATION_CAP_RATIO
    mapping = get_residue_mapping(aligned_pairs, target_len, template_len)
    result = xp.zeros((target_len, 3), dtype=xp.float32)
    cap_len = int(template_len * extrapolation_cap_ratio) if template_len > 0 else target_len
    for i in range(target_len):
        j = int(mapping[i])
        if 0 <= j < len(template_coords):
            result[i] = template_coords[j]
        elif len(template_coords) > 0:
            result[i] = template_coords[min(j, len(template_coords) - 1)]
    if target_len > cap_len and cap_len >= 3:
        tail_len = target_len - cap_len
        p0, p1 = result[cap_len - 2], result[cap_len - 1]
        direction = p1 - p0
        norm = float(xp.linalg.norm(direction))
        direction = direction / norm if norm >= 1e-6 else xp.array([1.0, 0.0, 0.0], dtype=xp.float32)
        aform = generate_aform_helix(tail_len, offset=0)
        aform[:, 2] *= (5.9 / 2.81)
        aform += result[cap_len - 1] - aform[0]
        result[cap_len:] = aform
    if target_len > SMOOTHING_WINDOW:
        w = SMOOTHING_WINDOW
        half = w // 2
        smoothed = result.copy()
        for i in range(half, target_len - half):
            smoothed[i] = result[i - half : i - half + w].mean(axis=0)
        result = smoothed
    if apply_kabsch and aligned_pairs and len(aligned_pairs) >= 3:
        t_indices = [ti for ti, si in aligned_pairs if 0 <= ti < target_len and 0 <= si < len(template_coords)]
        s_indices = [si for ti, si in aligned_pairs if 0 <= ti < target_len and 0 <= si < len(template_coords)]
        if len(t_indices) >= 3:
            P = result[xp.array(t_indices)]
            Q = template_coords[xp.array(s_indices)]
            if bool(xp.isfinite(P).all() and xp.isfinite(Q).all()):
                R, t, _ = kabsch_rmsd(P, Q)
                result = apply_superposition(result, R, t)
    return result

def predict_single_chain(chain_seq, target_len, train_seq_map, train_structures, allowed_template_ids=None, n_predictions=5, run_log=None):
    log_entries = []
    candidates = [c for c in kmer_prefilter(chain_seq, train_seq_map, train_structures, allowed_ids=allowed_template_ids) if c in train_structures]
    if not candidates:
        helix = generate_aform_helix(target_len)
        return [helix] * n_predictions, [{"source": "aform", "template_id": None, "score": 0.0}] * n_predictions
    ranked = rank_templates_by_alignment(chain_seq, candidates, train_seq_map)
    if not ranked or ranked[0][1] < MIN_ALIGNMENT_SCORE_FOR_TEMPLATE:
        helix = generate_aform_helix(target_len)
        return [helix] * n_predictions, [{"source": "aform", "template_id": None, "score": 0.0}] * n_predictions
    predictions = []
    used_sources = set()
    for template_id, score, aligned_pairs in ranked:
        if len(predictions) >= n_predictions or template_id in used_sources:
            continue
        used_sources.add(template_id)
        template_data = train_structures[template_id]
        template_coords = template_data["coords"]
        coords = transfer_coords_with_alignment(template_coords, aligned_pairs, target_len, len(template_coords), apply_kabsch=True) if aligned_pairs else _fallback_transfer(template_coords, target_len)
        predictions.append(_sanitize_coords(coords))
        log_entries.append({"source": "template", "template_id": template_id, "score": float(score)})
    if len(predictions) < n_predictions and ranked:
        best_id, best_score, best_aligned = ranked[0][0], ranked[0][1], ranked[0][2]
        best_data = train_structures[best_id]
        if best_data["n_copies"] > 1:
            for copy_id, copy_coords in best_data["all_copies"].items():
                if len(predictions) >= n_predictions:
                    break
                key = f"{best_id}_copy{copy_id}"
                if key in used_sources:
                    continue
                used_sources.add(key)
                coords = transfer_coords_with_alignment(copy_coords, best_aligned, target_len, len(copy_coords), apply_kabsch=True) if best_aligned else _fallback_transfer(copy_coords, target_len)
                predictions.append(_sanitize_coords(coords))
                log_entries.append({"source": "copy", "template_id": best_id, "score": float(best_score)})
    try:
        rng = xp.random.default_rng(SEED)
    except AttributeError:
        rng = np.random.default_rng(SEED)
    while len(predictions) < n_predictions:
        base = predictions[0].copy()
        noise = rng.normal(0, PERTURBATION_SIGMA, base.shape)
        noise = xp.asarray(noise) if not hasattr(noise, "get") else noise
        predictions.append(_sanitize_coords(base + noise))
        log_entries.append({"source": "perturb", "template_id": None, "score": 0.0})
    return predictions, log_entries

def _allowed_templates_for_cutoff(train_sequences_df, test_cutoff_str):
    test_d = _parse_cutoff(test_cutoff_str)
    if test_d is None:
        return None
    allowed = set()
    for _, row in train_sequences_df.iterrows():
        t = _parse_cutoff(row.get("temporal_cutoff"))
        if t is not None and t < test_d:
            allowed.add(row["target_id"])
    return allowed if allowed else None

def predict_with_templates(test_sequences_df, train_sequences_df, train_structures, submission_targets, n_predictions=5):
    train_seq_map = dict(zip(train_sequences_df["target_id"], train_sequences_df["sequence"]))
    predictions = {}
    for _, row in tqdm(test_sequences_df.iterrows(), total=len(test_sequences_df), desc="Predicting"):
        target_id = row["target_id"]
        test_seq = row["sequence"]
        stoich_str = row["stoichiometry"]
        all_sequences_str = row.get("all_sequences", None)
        if target_id not in submission_targets:
            continue
        n_residues = len(submission_targets[target_id])
        stoich = parse_stoichiometry(stoich_str)
        total_copies = sum(c for _, c in stoich)
        unique_chain_labels = set(l for l, _ in stoich)
        allowed = _allowed_templates_for_cutoff(train_sequences_df, row.get("temporal_cutoff"))
        chains = get_chain_sequences(test_seq, stoich, all_sequences_str)
        if total_copies > 1 and len(unique_chain_labels) == 1:
            _, n_copies = stoich[0]
            chain_seq = chains[0][2]
            chain_len = len(chain_seq)
            chain_preds, log_entries = predict_single_chain(chain_seq, chain_len, train_seq_map, train_structures, allowed_template_ids=allowed, n_predictions=n_predictions)
            full_preds = []
            for pred in chain_preds:
                assembled = generate_symmetric_copies(pred, n_copies)
                if len(assembled) >= n_residues:
                    full_preds.append(asnumpy(assembled[:n_residues]))
                else:
                    padded = xp.zeros((n_residues, 3), dtype=xp.float32)
                    padded[:len(assembled)] = assembled
                    full_preds.append(asnumpy(padded))
            predictions[target_id] = full_preds
        elif total_copies > 1 and len(unique_chain_labels) > 1:
            all_chain_preds = [predict_single_chain(c[2], len(c[2]), train_seq_map, train_structures, allowed_template_ids=allowed, n_predictions=n_predictions)[0] for c in chains]
            full_preds = [asnumpy(assemble_multimer([c[pred_idx] for c in all_chain_preds], stoich, n_residues)) for pred_idx in range(n_predictions)]
            predictions[target_id] = full_preds
        else:
            preds, _ = predict_single_chain(test_seq, n_residues, train_seq_map, train_structures, allowed_template_ids=allowed, n_predictions=n_predictions)
            predictions[target_id] = [asnumpy(p) for p in preds]
    return predictions

In [ ]:
# --- Submission ---
def _sanity_check_coords(df, coord_cols, sample_size=200):
    warnings = []
    if df[coord_cols].isna().any().any():
        warnings.append("Found NaN in coordinates")
    if not np.isfinite(df[coord_cols].values).all():
        warnings.append("Found non-finite values in coordinates")
    rng = np.random.default_rng(42)
    indices = rng.integers(0, max(1, len(df) - 1), size=min(sample_size, len(df)))
    for i in indices:
        if i + 1 >= len(df):
            continue
        row1, row2 = df.iloc[i], df.iloc[i + 1]
        if row1["ID"].rsplit("_", 1)[0] != row2["ID"].rsplit("_", 1)[0]:
            continue
        for pred_idx in range(1, 6):
            x1 = [row1[f"x_{pred_idx}"], row1[f"y_{pred_idx}"], row1[f"z_{pred_idx}"]]
            x2 = [row2[f"x_{pred_idx}"], row2[f"y_{pred_idx}"], row2[f"z_{pred_idx}"]]
            d = np.sqrt(sum((a - b) ** 2 for a, b in zip(x1, x2)))
            if d > 25:
                warnings.append("Very large C1'-C1' distance in sample")
                break
        if warnings and "Very large" in str(warnings[-1]):
            break
    return warnings

def create_submission(predictions, submission_targets, output_path):
    rows = []
    for target_id, residues in submission_targets.items():
        if target_id not in predictions:
            for res in residues:
                rows.append({"ID": res["ID"], "resname": res["resname"], "resid": res["resid"], **{f"x_{j+1}": 0.0 for j in range(5)}, **{f"y_{j+1}": 0.0 for j in range(5)}, **{f"z_{j+1}": 0.0 for j in range(5)}})
            continue
        pred_list = predictions[target_id]
        n_residues = len(residues)
        for i, res in enumerate(residues):
            row = {"ID": res["ID"], "resname": res["resname"], "resid": res["resid"]}
            for j in range(5):
                if j < len(pred_list) and i < len(pred_list[j]):
                    row[f"x_{j+1}"] = round(float(pred_list[j][i, 0]), 3)
                    row[f"y_{j+1}"] = round(float(pred_list[j][i, 1]), 3)
                    row[f"z_{j+1}"] = round(float(pred_list[j][i, 2]), 3)
                else:
                    row[f"x_{j+1}"] = row[f"y_{j+1}"] = row[f"z_{j+1}"] = 0.0
            rows.append(row)
    columns = ["ID", "resname", "resid"] + [c for j in range(1, 6) for c in [f"x_{j}", f"y_{j}", f"z_{j}"]]
    df = pd.DataFrame(rows, columns=columns)
    coord_cols = [c for c in columns if c.startswith(("x_", "y_", "z_"))]
    df[coord_cols] = df[coord_cols].fillna(0.0)
    for w in _sanity_check_coords(df, coord_cols):
        print("  Sanity:", w)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    return df

def validate_submission(submission_path, sample_submission_path):
    sub = pd.read_csv(submission_path)
    sample = pd.read_csv(sample_submission_path)
    errors = []
    if list(sub.columns) != list(sample.columns):
        errors.append("Column mismatch")
    if len(sub) != len(sample):
        errors.append(f"Row count mismatch: {len(sub)} vs {len(sample)}")
    if not sub["ID"].equals(sample["ID"]):
        errors.append("ID mismatch")
    if sub[[c for c in sub.columns if c.startswith(('x_','y_','z_'))]].isna().any().any():
        errors.append("NaN in coordinates")
    if errors:
        print("VALIDATION FAILED:", errors)
        return False
    print("Submission validation PASSED")
    return True

In [ ]:
# --- Load data ---
# Resolve data path (Kaggle may use a subfolder or different mount name)
if not (INPUT_DIR / "train_sequences.csv").exists():
    for _p in INPUT_DIR.rglob("train_sequences.csv"):
        INPUT_DIR = _p.parent
        break
if not (INPUT_DIR / "train_sequences.csv").exists() and Path("/kaggle/input").exists():
    for _p in Path("/kaggle/input").rglob("train_sequences.csv"):
        INPUT_DIR = _p.parent
        break
print("Loading data from", INPUT_DIR)
train_seq_df = load_train_sequences(INPUT_DIR)
train_labels_df = load_train_labels(INPUT_DIR)
train_structures = build_train_structure_lookup(train_labels_df)
del train_labels_df
test_seq_df = load_test_sequences(INPUT_DIR)
sample_sub_df = load_sample_submission(INPUT_DIR)
submission_targets = parse_submission_targets(sample_sub_df)
print(f"Train structures: {len(train_structures)}, Test targets: {len(submission_targets)}, Total residues: {len(sample_sub_df)}")

In [ ]:
# --- Run prediction and create submission ---
n_predictions = 5
predictions = predict_with_templates(test_seq_df, train_seq_df, train_structures, submission_targets, n_predictions=n_predictions)
# Write to Kaggle working dir; competition requires output file named exactly submission.csv
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = OUTPUT_DIR / "submission.csv"
create_submission(predictions, submission_targets, submission_path)
# Also write to cwd so Kaggle's submission detector finds it
import shutil
shutil.copy(str(submission_path), "submission.csv")
validate_submission(str(submission_path), str(INPUT_DIR / "sample_submission.csv"))
print("Done. Submission file:", submission_path)
print("Also written to:", Path("submission.csv").resolve())

**How to use on Kaggle**

1. Go to [Stanford RNA 3D Folding Part 2](https://www.kaggle.com/competitions/stanford-rna-3d-folding-2), join the competition, and open **Code**.
2. Create a new notebook and copy this notebook’s cells (or upload this `.ipynb`).
3. Ensure the competition dataset is attached (right sidebar → **Add data** → competition data).
4. **Run All** and wait until every cell finishes (no errors). The notebook writes `submission.csv` to `/kaggle/working/` and to the working directory.
5. **Save Version** → **Save & Run All (Commit)**. Wait for the run to complete successfully.
6. **Submit**: Competition page → **Submit Predictions** → **Notebook** → select a **Version** that completed without errors. If you see "does not output this file", that run failed or did not finish; pick a successful run or re-run.